# Palm Tree Detection — Clean ML Notebook (YOLOv8)
Notebook ini dipecah menjadi beberapa bagian: setup, persiapan dataset, augmentasi, training, evaluasi, inferensi, dan ekspor.

In [1]:
# 1. Setup
import os
import yaml
import random
import shutil
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO
import torch

# Basic PyTorch / CUDA checks
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version (built): {torch.version.cuda}")
print(f"CUDA device count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    try:
        print(f"Device: {torch.cuda.get_device_name(0)}")
    except Exception as e:
        print('Device name unavailable:', e)
    try:
        print(f"cuDNN version: {torch.backends.cudnn.version()}")
    except Exception:
        pass

# OpenCV is not required in this notebook.
# Matplotlib is used for visualization and annotation to avoid native build issues on Windows.

PyTorch: 2.5.1+cu121
CUDA available: True
CUDA version (built): 12.1
CUDA device count: 1
Device: NVIDIA GeForce RTX 3060
cuDNN version: 90100


In [2]:
# 2. Dataset Preparation
# Since dataset is now in notebook/ folder, use relative path
DATA_ROOT = Path("dataset")
print('DATA_ROOT:', DATA_ROOT.resolve())
print('Images in train:', len(list((DATA_ROOT / 'images' / 'train').glob('*'))))

def create_dataset_structure():
    for split in ['train', 'val', 'test']:
        (DATA_ROOT / 'images' / split).mkdir(parents=True, exist_ok=True)
        (DATA_ROOT / 'labels' / split).mkdir(parents=True, exist_ok=True)

    yaml_content = {
        'path': 'dataset',
        'train': 'images/train',
        'val': 'images/val',
        'test': 'images/test',
        'nc': 5,
        'names': ['Dead', 'Healthy', 'Grass', 'Small', 'Yellow']
    }

    with open(DATA_ROOT / 'data.yaml', 'w') as f:
        yaml.dump(yaml_content, f)

    print("Dataset structure created")
    print("data.yaml content:")
    with open(DATA_ROOT / 'data.yaml') as f:
        print(f.read())


def split_dataset(source_img_dir, source_lbl_dir, ratios=(0.7, 0.2, 0.1), seed=42):
    random.seed(seed)

    images = sorted([
        f for f in Path(source_img_dir).glob('*')
        if f.suffix.lower() in ['.jpg', '.png', '.jpeg']
    ])

    random.shuffle(images)

    n = len(images)
    n_train = int(n * ratios[0])
    n_val = int(n * ratios[1])

    splits = {
        'train': images[:n_train],
        'val': images[n_train:n_train + n_val],
        'test': images[n_train + n_val:]
    }

    for split, imgs in splits.items():
        for img in imgs:
            dst_img = DATA_ROOT / 'images' / split / img.name
            dst_img.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(str(img), str(dst_img))

            lbl = Path(source_lbl_dir) / img.with_suffix('.txt').name
            if lbl.exists():
                dst_lbl = DATA_ROOT / 'labels' / split / lbl.name
                dst_lbl.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(str(lbl), str(dst_lbl))

        print(f"{split}: {len(imgs)} images")


def visualize_samples(img_dir, lbl_dir, n=4, classes=None):
    if classes is None:
        classes = ['Dead', 'Healthy', 'Grass', 'Small', 'Yellow']

    colors = {0: (34, 197, 94), 1: (239, 68, 68)}
    imgs = list(Path(img_dir).glob('*'))[:n]

    if not imgs:
        print("No images found to visualize.")
        return

    fig, axes = plt.subplots(1, len(imgs), figsize=(5 * len(imgs), 5))
    if len(imgs) == 1:
        axes = [axes]

    for ax, img_path in zip(axes, imgs):
        img = plt.imread(str(img_path))
        if img.dtype.kind == 'f':
            img = (img * 255).clip(0, 255).astype(np.uint8)

        h, w = img.shape[:2]
        lbl_path = Path(lbl_dir) / img_path.with_suffix('.txt').name

        ax.imshow(img)
        ax.axis('off')

        if lbl_path.exists():
            with open(lbl_path) as f:
                for line in f:
                    parts = line.split()
                    if len(parts) < 5:
                        continue

                    cls, cx, cy, bw, bh = map(float, parts[:5])
                    x1 = int((cx - bw / 2) * w)
                    y1 = int((cy - bh / 2) * h)
                    x2 = int((cx + bw / 2) * w)
                    y2 = int((cy + bh / 2) * h)

                    color = tuple(channel / 255 for channel in colors.get(int(cls), (255, 255, 255)))
                    label = classes[int(cls)] if int(cls) < len(classes) else str(int(cls))

                    rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=color, linewidth=2)
                    ax.add_patch(rect)
                    ax.text(
                        x1,
                        max(0, y1 - 4),
                        label,
                        color=color,
                        fontsize=9,
                        bbox=dict(facecolor='black', alpha=0.35, pad=1, edgecolor='none'),
                    )

    plt.tight_layout()
    plt.show()


create_dataset_structure()
# split_dataset('raw_images', 'raw_labels')
# visualize_samples(str(DATA_ROOT / 'images' / 'train'), str(DATA_ROOT / 'labels' / 'train'))


DATA_ROOT: D:\Website Development\palm-tree-detection\notebook\dataset
Images in train: 1803
Dataset structure created
data.yaml content:
names:
- Dead
- Healthy
- Grass
- Small
- Yellow
nc: 5
path: dataset
test: images/test
train: images/train
val: images/val



In [3]:
# 2b. Dataset Validation (COCO -> YOLO)
import json
from collections import defaultdict

def coco_bbox_to_yolo(bbox, img_w, img_h):
    x, y, w, h = bbox
    cx = (x + w / 2) / img_w
    cy = (y + h / 2) / img_h
    return cx, cy, w / img_w, h / img_h


def prepare_yolo_from_coco(coco_path, images_src, images_dst, labels_dst, copy_images=True):
    with open(coco_path, "r", encoding="utf-8") as f:
        coco = json.load(f)

    categories = sorted(coco.get("categories", []), key=lambda c: c["id"])
    category_id_to_index = {c["id"]: i for i, c in enumerate(categories)}
    names = [c["name"] for c in categories]

    images = {img["id"]: img for img in coco.get("images", [])}
    labels_by_file = defaultdict(list)

    for ann in coco.get("annotations", []):
        if ann.get("iscrowd", 0) == 1:
            continue
        img = images.get(ann.get("image_id"))
        if not img:
            continue
        bbox = ann.get("bbox")
        if not bbox:
            continue

        img_w = img.get("width", 0)
        img_h = img.get("height", 0)
        if img_w <= 0 or img_h <= 0:
            continue

        cls = category_id_to_index.get(ann.get("category_id"))
        if cls is None:
            continue

        x, y, w, h = bbox
        if w <= 0 or h <= 0:
            continue

        cx, cy, bw, bh = coco_bbox_to_yolo(bbox, img_w, img_h)
        labels_by_file[img["file_name"]].append(
            f"{cls} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}"
        )

    labels_dst.mkdir(parents=True, exist_ok=True)
    images_dst.mkdir(parents=True, exist_ok=True)

    for file_name, lines in labels_by_file.items():
        label_path = labels_dst / Path(file_name).with_suffix(".txt").name
        label_path.write_text("\n".join(lines))

    if copy_images:
        for img in images.values():
            src = Path(images_src) / img["file_name"]
            dst = images_dst / img["file_name"]
            if not src.exists():
                continue
            if not dst.exists():
                dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(src, dst)

    return names


def ensure_yolo_dataset():
    train_images = DATA_ROOT / "images/train"
    train_labels = DATA_ROOT / "labels/train"
    val_images = DATA_ROOT / "images/val"
    val_labels = DATA_ROOT / "labels/val"

    has_train = train_images.exists() and any(train_images.glob("*"))
    has_labels = train_labels.exists() and any(train_labels.glob("*.txt"))

    if has_train and has_labels:
        print("YOLO dataset already prepared.")
        return

    coco_train = DATA_ROOT / "annotations/instances_train2017.json"
    coco_val = DATA_ROOT / "annotations/instances_val2017.json"
    names = []

    if coco_train.exists() and (DATA_ROOT / "train2017").exists():
        names = prepare_yolo_from_coco(
            coco_train,
            DATA_ROOT / "train2017",
            train_images,
            train_labels,
            copy_images=True,
        )

    if coco_val.exists() and (DATA_ROOT / "val2017").exists():
        names_val = prepare_yolo_from_coco(
            coco_val,
            DATA_ROOT / "val2017",
            val_images,
            val_labels,
            copy_images=True,
        )
        if not names:
            names = names_val

    if not names:
        print("Could not prepare YOLO dataset. Check dataset paths.")
        return

    data_yaml = {
        "path": str(DATA_ROOT),
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "nc": len(names),
        "names": names,
    }

    with open(DATA_ROOT / "data.yaml", "w", encoding="utf-8") as f:
        yaml.dump(data_yaml, f)

    print("YOLO dataset prepared.")


ensure_yolo_dataset()

YOLO dataset already prepared.


In [4]:
# 3. Data Augmentation
import albumentations as A

augment_pipeline = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(p=0.4),
    A.HueSaturationValue(p=0.3),
    A.GaussianBlur(blur_limit=3, p=0.2),
    A.RandomFog(fog_coef_lower=0.1, fog_coef_upper=0.3, p=0.1),
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

print(f"Augmentation pipeline: {len(augment_pipeline.transforms)} transforms")

Augmentation pipeline: 7 transforms


C:\Users\wahyu\AppData\Local\Temp\ipykernel_7256\278857403.py:11: UserWarning: Argument(s) 'fog_coef_lower, fog_coef_upper' are not valid for transform RandomFog
  A.RandomFog(fog_coef_lower=0.1, fog_coef_upper=0.3, p=0.1),


In [7]:
# 4. Model Training
import torch

# Clear GPU cache before training
torch.cuda.empty_cache()

model = YOLO('yolov8s.pt')

results = model.train(
    data='dataset/data.yaml',
    epochs=100,
    imgsz=640,
    batch=8,  # Reduced from 16 for 8GB GPU
    workers=0,  # IMPORTANT: Set to 0 on Windows to avoid DataLoader crashes
    lr0=0.01,
    lrf=0.001,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,

    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10,
    flipud=0.3,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,

    project='runs/palm_detection',
    name='yolov8s_exp1',
    save=True,
    plots=True,
    device='0',
    patience=10,  # Early stopping after 10 epochs without improvement
    close_mosaic=10  # Disable mosaic augmentation for last 10 epochs
    )

New https://pypi.org/project/ultralytics/8.4.51 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.50  Python-3.12.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3060, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset/data.yaml, degrees=10, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.3, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.001, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8

In [ ]:
# 5. Evaluation
best_model = YOLO('runs/palm_detection/yolov8s_exp1/weights/best.pt')

metrics = best_model.val(
    data='dataset/data.yaml',
    split='test',
    conf=0.5,
    iou=0.5
)

print(f"mAP@0.5      : {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95 : {metrics.box.map:.4f}")
print(f"Precision    : {metrics.box.mp:.4f}")
print(f"Recall       : {metrics.box.mr:.4f}")

from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

y_true, y_pred = [], []

for img_path in Path('dataset/images/test').glob('*.jpg'):
    res = best_model.predict(img_path, conf=0.5, verbose=False)[0]

    for box in res.boxes:
        y_pred.append(int(box.cls))

    lbl_path = Path('dataset/labels/test') / img_path.with_suffix('.txt').name

    if lbl_path.exists():
        with open(lbl_path) as f:
            for line in f:
                y_true.append(int(line.split()[0]))

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['palm', 'non_palm'],
            yticklabels=['palm', 'non_palm'])
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

print(classification_report(y_true, y_pred,
      target_names=['palm_tree', 'non_palm']))

In [ ]:
# 6. Inference & Counting
def detect_and_count(image_path, model, conf=0.5, save_path=None):
    img = plt.imread(str(image_path))
    if img.dtype.kind == 'f':
        img = (img * 255).clip(0, 255).astype(np.uint8)

    results = model.predict(img, conf=conf, iou=0.4, verbose=False)[0]

    palm_count = 0
    non_palm_count = 0
    palette = {
        0: (34, 197, 94),
        1: (239, 68, 68),
    }

    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(img)
    ax.axis('off')

    for box in results.boxes:
        cls = int(box.cls)
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        color = tuple(channel / 255 for channel in palette.get(cls, (255, 255, 255)))
        label = results.names.get(cls, str(cls)) if hasattr(results, 'names') else str(cls)
        confidence = float(box.conf)

        if cls == 0:
            palm_count += 1
        elif cls == 1:
            non_palm_count += 1

        rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=color, linewidth=2)
        ax.add_patch(rect)
        ax.text(
            x1,
            max(0, y1 - 4),
            f"{label} {confidence:.2f}",
            color=color,
            fontsize=9,
            bbox=dict(facecolor='black', alpha=0.35, pad=1, edgecolor='none'),
        )

    ax.text(10, 30, f"Palm trees: {palm_count}", color=(34/255, 197/255, 94/255), fontsize=12, bbox=dict(facecolor='black', alpha=0.35, pad=2, edgecolor='none'))
    ax.text(10, 55, f"Non-palm : {non_palm_count}", color=(239/255, 68/255, 68/255), fontsize=12, bbox=dict(facecolor='black', alpha=0.35, pad=2, edgecolor='none'))

    fig.canvas.draw()
    annotated = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8)
    annotated = annotated.reshape(fig.canvas.get_width_height()[::-1] + (4,))[:, :, :3].copy()

    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, bbox_inches='tight', pad_inches=0)

    plt.close(fig)
    return annotated, palm_count, non_palm_count


# Example inference
# img, n_palm, n_non = detect_and_count(
#     'dataset/images/test/sample.jpg',
#     best_model,
#     save_path='output/detected.jpg'
# )

In [ ]:
# 7. Export Model
best_model.export(
    format='onnx',
    imgsz=640,
    simplify=True,
    dynamic=False,
    opset=17
)

print("Model exported to ONNX")
print("Path: runs/palm_detection/yolov8s_exp1/weights/best.onnx")